# MyanNet — Lightweight CNN for Burmese Handwritten Digit Recognition

**Dataset:** Burmese Handwritten Digit Dataset (BHDD)  
**Task:** 10-class digit classification (၀–၉)  
**Architecture:** Depthwise Separable CNN with Global Average Pooling  
**Best Result:** 99.55% test accuracy · 5,418 trainable parameters

---

### Notebook Structure
| Section | Description |
|---|---|
| 1 | Environment Setup & Reproducibility |
| 2 | Data Loading & Preprocessing |
| 3 | Data Augmentation |
| 4 | Model Definitions (Baseline → MyanNet) |
| 5 | Hyperparameter Search (Optuna) |
| 6 | Final MyanNet Training |
| 7 | 5-Fold Stratified Cross-Validation |
| 8 | Evaluation & Classification Report |
| 9 | Confusion Matrix & Training Curves |
| 10 | Model Comparison |
| 11 | TFLite Export & Inference Benchmarking |
| 12 | Results Summary |

---

## Section 1 — Environment Setup & Reproducibility

In [1]:
import os
import sys
import json
import time
import random
import pickle
import datetime
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility: set ALL seeds before importing TensorFlow ──
SEED = 42
os.environ['PYTHONHASHSEED']    = str(SEED)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import matplotlib
matplotlib.use('Agg')          # non-interactive backend (safe on clusters)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import tensorflow as tf
tf.random.set_seed(SEED)

print(f'Python     : {sys.version.split()[0]}')
print(f'TensorFlow : {tf.__version__}')
print(f'NumPy      : {np.__version__}')
print(f'GPUs found : {len(tf.config.list_physical_devices("GPU"))}')
print(f'Seed       : {SEED}')

2026-04-13 07:12:07.236116: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776064327.429175      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776064327.485498      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776064327.942321      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776064327.942350      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776064327.942352      24 computation_placer.cc:177] computation placer alr

Python     : 3.12.12
TensorFlow : 2.19.0
NumPy      : 2.0.2
GPUs found : 1
Seed       : 42


In [2]:
# ── Check / install Optuna ──
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    print(f'Optuna : {optuna.__version__}')
except ImportError:
    print('Optuna not found — installing...')
    os.system('pip install optuna -q')
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    print(f'Optuna installed: {optuna.__version__}')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
print('scikit-learn : OK')

Optuna : 4.8.0
scikit-learn : OK


---

## Section 2 — Data Loading & Preprocessing

In [3]:
from tensorflow.keras.utils import to_categorical

DATA_PATH   = '/kaggle/input/datasets/ahmaungoo/bhdd-set/data.pkl'
NUM_CLASSES = 10

with open(DATA_PATH, 'rb') as f:
    dataset = pickle.load(f)

def extract_arrays(split):
    """Convert list-of-dicts to (images, labels) numpy arrays."""
    X = np.array([entry['image'] for entry in split], dtype=np.float32)
    y = np.array([entry['label'] for entry in split], dtype=np.int32)
    return X, y

X_train_raw, y_train = extract_arrays(dataset['trainDataset'])
X_test_raw,  y_test  = extract_arrays(dataset['testDataset'])

# ── Normalise to [0, 1] and reshape for CNN ──
X_train = (X_train_raw / 255.0).reshape(-1, 28, 28, 1)
X_test  = (X_test_raw  / 255.0).reshape(-1, 28, 28, 1)

# ── One-hot labels (required for label_smoothing loss) ──
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

print('─' * 40)
print(f'Training samples  : {len(X_train):,}')
print(f'Test samples      : {len(X_test):,}')
print(f'Image shape       : {X_train.shape[1:]}')
print(f'Classes           : {NUM_CLASSES}  (labels {y_train.min()}–{y_train.max()})')
print(f'Pixel range       : [{X_train.min():.1f}, {X_train.max():.1f}]')
print('─' * 40)

────────────────────────────────────────
Training samples  : 60,000
Test samples      : 27,561
Image shape       : (28, 28, 1)
Classes           : 10  (labels 0–9)
Pixel range       : [0.0, 1.0]
────────────────────────────────────────


In [4]:
# ── Visualise one sample per class ──
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

for digit in range(10):
    idx = np.where(y_train == digit)[0][0]
    axes[digit].imshow(X_train[idx].reshape(28, 28), cmap='gray')
    axes[digit].set_title(f'Label: {digit}', fontsize=11)
    axes[digit].axis('off')

fig.suptitle('BHDD — One Sample per Class (0–9)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sample_images.png')

Saved: sample_images.png


In [5]:
# ── Class distribution ──
train_counts = np.bincount(y_train)
test_counts  = np.bincount(y_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, counts, title in zip(axes,
                              [train_counts, test_counts],
                              ['Training Set', 'Test Set']):
    bars = ax.bar(range(10), counts, color='steelblue', edgecolor='white')
    for bar, v in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{v:,}', ha='center', va='bottom', fontsize=8)
    ax.set_title(f'{title} — Class Distribution', fontsize=12)
    ax.set_xlabel('Digit Class (0–9)')
    ax.set_ylabel('Count')
    ax.set_xticks(range(10))
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: class_distribution.png')

Saved: class_distribution.png


---

## Section 3 — Data Augmentation

Augmentation is applied **only during training** to improve generalisation.  
Parameters were chosen to reflect realistic handwriting variation:

| Parameter | Value | Rationale |
|---|---|---|
| `rotation_range` | 15° | Slight pen tilt |
| `width_shift_range` | 10% | Horizontal offset |
| `height_shift_range` | 10% | Vertical offset |
| `zoom_range` | 10% | Writing pressure / size variance |
| `shear_range` | 10% | Cursive slant |

In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range    = 15,
    width_shift_range = 0.1,
    height_shift_range= 0.1,
    zoom_range        = 0.1,
    shear_range       = 0.1
)
datagen.fit(X_train)

print('ImageDataGenerator configured.')

# ── Preview augmented samples ──
sample_img = X_train[0:1]
aug_iter   = datagen.flow(sample_img, batch_size=1)

fig, axes = plt.subplots(1, 8, figsize=(14, 2))
axes[0].imshow(sample_img[0].reshape(28, 28), cmap='gray')
axes[0].set_title('Original', fontsize=9)
axes[0].axis('off')

for i in range(1, 8):
    aug_img = next(aug_iter)[0]
    axes[i].imshow(aug_img.reshape(28, 28), cmap='gray')
    axes[i].set_title(f'Aug {i}', fontsize=9)
    axes[i].axis('off')

plt.suptitle('Data Augmentation Preview', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('augmentation_preview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: augmentation_preview.png')

ImageDataGenerator configured.
Saved: augmentation_preview.png


---

## Section 4 — Model Definitions

Three architectures are compared:

| # | Model | Key Feature |
|---|---|---|
| 1 | **Baseline CNN** | Standard Conv → Flatten → Dense |
| 2 | **GAP-BN CNN** | Conv + BatchNorm + Global Average Pooling |
| 3 | **MyanNet** *(proposed)* | Depthwise Separable Conv + GAP + Dropout |

In [7]:
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, DepthwiseConv2D, BatchNormalization,
    MaxPooling2D, GlobalAveragePooling2D,
    Dropout, Dense, Flatten
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import (
    ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
)

# ── Multi-GPU strategy (auto-detects; falls back to single GPU/CPU) ──
strategy  = tf.distribute.MirroredStrategy()
NUM_GPUS  = strategy.num_replicas_in_sync
BATCH_SIZE = 64 * max(1, NUM_GPUS)
print(f'MirroredStrategy replicas : {NUM_GPUS}')
print(f'Effective batch size      : {BATCH_SIZE}')

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0',)
MirroredStrategy replicas : 1
Effective batch size      : 64


I0000 00:00:1776064353.882393      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [8]:
# ── Model 1: Baseline CNN ──
def build_baseline():
    model = Sequential([
        Input(shape=(28, 28, 1)),
        Conv2D(32, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(10, activation='softmax')
    ], name='Baseline_CNN')
    model.compile(
        optimizer=Adam(),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# ── Model 2: GAP + BatchNorm CNN ──
def build_gap_bn():
    model = Sequential([
        Input(shape=(28, 28, 1)),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        GlobalAveragePooling2D(),
        Dense(32, activation='relu'),
        Dense(10, activation='softmax')
    ], name='GAP_BN_CNN')
    model.compile(
        optimizer=Adam(),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# ── Model 3: MyanNet (proposed) ──
def build_myannet(filters1=32, filters2=64, dropout=0.3,
                  learning_rate=1e-3, dense_units=32):
    """
    MyanNet — Lightweight Depthwise-Separable CNN.
    Block 1 : Standard Conv2D   → BN → MaxPool
    Block 2 : DepthwiseConv2D   → BN → Pointwise Conv2D → BN → MaxPool
    Head    : GAP → Dropout → Dense → Softmax
    """
    model = Sequential([
        Input(shape=(28, 28, 1)),

        # Block 1 — standard conv
        Conv2D(filters1, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),                         # → 14×14

        # Block 2 — depthwise separable conv
        DepthwiseConv2D((3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(filters2, (1, 1), activation='relu'),  # pointwise
        BatchNormalization(),
        MaxPooling2D((2, 2)),                         # → 7×7

        # Head
        GlobalAveragePooling2D(),                     # → filters2-dim vector
        Dropout(dropout),
        Dense(dense_units, activation='relu'),
        Dense(10, activation='softmax')
    ], name='MyanNet')

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss=CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy']
    )
    return model

# ── Print MyanNet summary ──
with strategy.scope():
    demo = build_myannet()
demo.summary()
print(f'\nTrainable parameters: {demo.count_params():,}')

Model: "MyanNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 14, 14, 32)     │           320 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 14, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,674 (22.16 KB)

 Trainable params: 5,418 (21.16 KB)

 Non-trainable params: 256 (1.00 KB)


Trainable parameters: 5,674


---

## Section 5 — Hyperparameter Search with Optuna

**Search space:**

| Hyperparameter | Candidates |
|---|---|
| `filters1` | 16, 32, 64 |
| `filters2` | 32, 64, 128 |
| `dropout` | 0.1 – 0.5 (continuous) |
| `learning_rate` | 1e-4 – 1e-2 (log scale) |
| `dense_units` | 16, 32, 64 |

Each trial trains for up to 15 epochs with early stopping (patience=4) — a fast proxy for full training.

In [9]:
OPTUNA_TRIALS    = 30
OPTUNA_EPOCHS    = 15
OPTUNA_VAL_SPLIT = 0.1

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import optuna
import json
import pickle

val_size = int(len(X_train) * OPTUNA_VAL_SPLIT)
X_opt_tr, X_opt_val = X_train[val_size:], X_train[:val_size]
y_opt_tr, y_opt_val = y_train_cat[val_size:], y_train_cat[:val_size]

def optuna_objective(trial):
    params = {
        'filters1'      : trial.suggest_categorical('filters1',   [16, 32, 64]),
        'filters2'      : trial.suggest_categorical('filters2',   [32, 64, 128]),
        'dropout'       : trial.suggest_float('dropout',          0.1, 0.4),
        'learning_rate' : trial.suggest_float('learning_rate',    1e-4, 1e-2, log=True),
        'dense_units'   : trial.suggest_categorical('dense_units', [16, 32, 64]),
    }
    with strategy.scope():
        model = build_myannet(**params)

    aug = datagen.flow(X_opt_tr, y_opt_tr, batch_size=BATCH_SIZE, seed=SEED)
    callbacks = [
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-5, verbose=0),
        EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=0),
    ]
    history = model.fit(
        aug,
        steps_per_epoch = len(X_opt_tr) // BATCH_SIZE,
        epochs          = OPTUNA_EPOCHS,
        validation_data = (X_opt_val, y_opt_val),
        verbose         = 0,
        callbacks       = callbacks
    )
    return max(history.history.get('val_accuracy', [0]))

print(f'Starting Optuna search — {OPTUNA_TRIALS} trials...\n')

study = optuna.create_study(
    direction = 'maximize',
    sampler   = optuna.samplers.TPESampler(seed=SEED)
)
study.optimize(optuna_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True, n_jobs=1)

best_params = study.best_params
print(f'\nBest validation accuracy : {study.best_value:.4f}')
print(f'Best hyperparameters     : {best_params}')

with open('best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
with open("optuna_study.pkl", "wb") as f:
    pickle.dump(study, f)
print('Saved: best_params.json and optuna_study.pkl')

Starting Optuna search — 30 trials...



  0%|          | 0/30 [00:00<?, ?it/s]

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


I0000 00:00:1776064357.246022      66 cuda_dnn.cc:529] Loaded cuDNN version 91002


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).

Best validation accuracy : 0.9968
Best hyperparameters     : {'filters1': 64, 'filters2': 64, 'dropout': 0.19369249293034319, 'learning_rate': 0.003004836279033769, 'dense_units': 64}
Saved: best_params.json and optuna_study.pkl


In [10]:
# ── Optuna optimisation history ──
trial_values = [t.value for t in study.trials if t.value is not None]
best_so_far  = np.maximum.accumulate(trial_values)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(trial_values, marker='o', markersize=4,
             color='steelblue', alpha=0.6, label='Trial val_acc')
axes[0].plot(best_so_far, color='darkorange', linewidth=2, label='Best so far')
axes[0].set_title('Optuna — Validation Accuracy per Trial', fontsize=12)
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Val Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Hyperparameter importance (manual bar chart)
param_names = list(best_params.keys())
axes[1].barh(param_names, [1]*len(param_names), color='steelblue', alpha=0.7)
axes[1].set_title('Best Hyperparameter Values', fontsize=12)
axes[1].set_xlabel('Selected value')
for i, (k, v) in enumerate(best_params.items()):
    axes[1].text(0.05, i, f'{v}', va='center', fontsize=10, fontweight='bold')
axes[1].set_xlim([0, 1.5])

plt.suptitle('Optuna Hyperparameter Search Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('optuna_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: optuna_results.png')

Saved: optuna_results.png


---

## Section 6 — Final MyanNet Training

**Training configuration:**

| Setting | Value |
|---|---|
| Max epochs | 100 |
| Batch size | 64 |
| Loss | CategoricalCrossentropy (label_smoothing=0.1) |
| Augmentation | Yes (rotation, shift, zoom, shear) |
| `ReduceLROnPlateau` | factor=0.5, patience=4, min_lr=1e-6 |
| `EarlyStopping` | patience=10, restore_best_weights=True |
| `ModelCheckpoint` | Save best val_accuracy |

In [11]:
print('Building final MyanNet with best hyperparameters...')
print(json.dumps(best_params, indent=2))

with strategy.scope():
    final_model = build_myannet(**best_params)

final_model.summary()
print(f'\nTrainable parameters: {final_model.count_params():,}')

Building final MyanNet with best hyperparameters...
{
  "filters1": 64,
  "filters2": 64,
  "dropout": 0.19369249293034319,
  "learning_rate": 0.003004836279033769,
  "dense_units": 64
}


Model: "MyanNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_62 (Conv2D)              │ (None, 28, 28, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_93          │ (None, 28, 28, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_62 (MaxPooling2D) │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_31             │ (None, 14, 14, 64)     │           640 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_94          │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_63 (Conv2D)              │ (None, 14, 14, 64)     │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_95          │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_63 (MaxPooling2D) │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_31     │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_31 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_62 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_63 (Dense)                │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,018 (43.04 KB)

 Trainable params: 10,634 (41.54 KB)

 Non-trainable params: 384 (1.50 KB)


Trainable parameters: 11,018


In [12]:
callbacks = [
    ReduceLROnPlateau(
        monitor  = 'val_accuracy',
        factor   = 0.5,
        patience = 4,
        min_lr   = 1e-6,
        verbose  = 1
    ),
    EarlyStopping(
        monitor              = 'val_accuracy',
        patience             = 10,
        restore_best_weights = True,
        verbose              = 1
    ),
    ModelCheckpoint(
        'myannet_best.keras',
        monitor        = 'val_accuracy',
        save_best_only = True,
        verbose        = 1
    )
]

aug_gen = datagen.flow(X_train, y_train_cat,
                       batch_size=BATCH_SIZE, seed=SEED)

print('\nTraining started...')
history_final = final_model.fit(
    aug_gen,
    steps_per_epoch = len(X_train) // BATCH_SIZE,
    epochs          = 100,
    validation_data = (X_test, y_test_cat),
    callbacks       = callbacks,
    verbose         = 2
)
print('\nTraining complete.')


Training started...
Epoch 1/100

Epoch 1: val_accuracy improved from -inf to 0.57219, saving model to myannet_best.keras
937/937 - 24s - 26ms/step - accuracy: 0.9088 - loss: 0.7852 - val_accuracy: 0.5722 - val_loss: 1.4735 - learning_rate: 0.0030
Epoch 2/100

Epoch 2: val_accuracy did not improve from 0.57219
937/937 - 3s - 3ms/step - accuracy: 0.9844 - loss: 0.6328 - val_accuracy: 0.5582 - val_loss: 1.5345 - learning_rate: 0.0030
Epoch 3/100

Epoch 3: val_accuracy improved from 0.57219 to 0.98004, saving model to myannet_best.keras
937/937 - 22s - 23ms/step - accuracy: 0.9845 - loss: 0.5896 - val_accuracy: 0.9800 - val_loss: 0.6092 - learning_rate: 0.0030
Epoch 4/100

Epoch 4: val_accuracy improved from 0.98004 to 0.98113, saving model to myannet_best.keras
937/937 - 3s - 4ms/step - accuracy: 0.9844 - loss: 0.5751 - val_accuracy: 0.9811 - val_loss: 0.6050 - learning_rate: 0.0030
Epoch 5/100

Epoch 5: val_accuracy did not improve from 0.98113
937/937 - 22s - 24ms/step - accuracy: 0.98

---

## Section 7 — 5-Fold Stratified Cross-Validation

Each fold trains a **fresh** MyanNet (with best hyperparameters) on 4/5 of the training data, then evaluates on the **full held-out test set** — giving a robust estimate of generalisation.

In [13]:
from tensorflow.keras.models import load_model

skf            = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_accuracies = []

for fold_idx, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train), start=1):

    print(f'\n{"─"*50}')
    print(f'  Fold {fold_idx} / 5')
    print(f'{"─"*50}')

    X_fold_tr  = X_train[train_idx]
    X_fold_val = X_train[val_idx]
    y_fold_tr  = y_train_cat[train_idx]
    y_fold_val = y_train_cat[val_idx]

    with strategy.scope():
        fold_model = build_myannet(**best_params)

    fold_gen = datagen.flow(X_fold_tr, y_fold_tr,
                            batch_size=BATCH_SIZE, seed=SEED)

    fold_model.fit(
        fold_gen,
        steps_per_epoch = len(X_fold_tr) // BATCH_SIZE,
        epochs          = 50,
        validation_data = (X_fold_val, y_fold_val),
        callbacks       = [
            EarlyStopping(monitor='val_accuracy', patience=8,
                          restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_accuracy', factor=0.5,
                              patience=4, min_lr=1e-6)
        ],
        verbose = 0
    )

    _, fold_acc = fold_model.evaluate(X_test, y_test_cat, verbose=0)
    fold_accuracies.append(float(fold_acc))
    print(f'  Test accuracy : {fold_acc:.4f} ({fold_acc*100:.2f}%)')

mean_acc = np.mean(fold_accuracies)
std_acc  = np.std(fold_accuracies)
print(f'\n{"═"*50}')
print(f'  5-Fold Mean : {mean_acc:.4f} ± {std_acc:.4f}')
print(f'{"═"*50}')

kfold_results = {
    'fold_accuracies' : fold_accuracies,
    'mean_accuracy'   : float(mean_acc),
    'std_accuracy'    : float(std_acc)
}
with open('kfold_results.json', 'w') as f:
    json.dump(kfold_results, f, indent=2)
print('Saved: kfold_results.json')


──────────────────────────────────────────────────
  Fold 1 / 5
──────────────────────────────────────────────────
  Test accuracy : 0.9940 (99.40%)

──────────────────────────────────────────────────
  Fold 2 / 5
──────────────────────────────────────────────────
  Test accuracy : 0.9938 (99.38%)

──────────────────────────────────────────────────
  Fold 3 / 5
──────────────────────────────────────────────────
  Test accuracy : 0.9955 (99.55%)

──────────────────────────────────────────────────
  Fold 4 / 5
──────────────────────────────────────────────────
  Test accuracy : 0.9952 (99.52%)

──────────────────────────────────────────────────
  Fold 5 / 5
──────────────────────────────────────────────────
  Test accuracy : 0.9944 (99.44%)

══════════════════════════════════════════════════
  5-Fold Mean : 0.9946 ± 0.0006
══════════════════════════════════════════════════
Saved: kfold_results.json


In [14]:
# ── K-Fold results bar chart ──
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    [f'Fold {i+1}' for i in range(5)],
    [a * 100 for a in fold_accuracies],
    color='steelblue', edgecolor='white'
)
ax.axhline(mean_acc * 100, color='darkorange', linewidth=2,
           linestyle='--', label=f'Mean = {mean_acc*100:.2f}%')
ax.fill_between(range(-1, 6),
                (mean_acc - std_acc) * 100,
                (mean_acc + std_acc) * 100,
                alpha=0.15, color='darkorange', label=f'±1 std = {std_acc*100:.2f}%')

for bar, acc in zip(bars, fold_accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{acc*100:.2f}%', ha='center', va='bottom', fontsize=10)

ax.set_ylim([min(fold_accuracies)*100 - 0.5, 100.2])
ax.set_ylabel('Test Accuracy (%)', fontsize=11)
ax.set_title('MyanNet — 5-Fold Stratified Cross-Validation', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('kfold_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: kfold_results.png')

Saved: kfold_results.png


---

## Section 8 — Evaluation & Classification Report

In [15]:
from tensorflow.keras.models import load_model

# ── Load best checkpoint ──
best_model = load_model('myannet_best.keras')

test_loss, test_acc = best_model.evaluate(
    X_test, y_test_cat, batch_size=256, verbose=0
)

y_pred_probs = best_model.predict(X_test, batch_size=256, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)

print('═' * 55)
print('  MyanNet — Final Test Set Results')
print('═' * 55)
print(f'  Test accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print(f'  Test loss     : {test_loss:.4f}')
print('═' * 55)

═══════════════════════════════════════════════════════
  MyanNet — Final Test Set Results
═══════════════════════════════════════════════════════
  Test accuracy : 0.9949  (99.49%)
  Test loss     : 0.5243
═══════════════════════════════════════════════════════


In [16]:
# ── Per-class classification report ──
print('═' * 55)
print('  CLASSIFICATION REPORT — MyanNet')
print('═' * 55)
print(classification_report(
    y_test, y_pred,
    target_names=[str(i) for i in range(10)],
    digits=4
))

═══════════════════════════════════════════════════════
  CLASSIFICATION REPORT — MyanNet
═══════════════════════════════════════════════════════
              precision    recall  f1-score   support

           0     0.9996    0.9901    0.9948      6856
           1     0.9946    0.9951    0.9949      6163
           2     1.0000    0.9959    0.9979      4339
           3     0.9963    0.9977    0.9970      3480
           4     0.9949    1.0000    0.9975      2162
           5     0.9854    0.9991    0.9922      2237
           6     0.9727    0.9980    0.9852       500
           7     0.9918    0.9869    0.9893       610
           8     0.9762    0.9964    0.9862       825
           9     0.9797    0.9949    0.9872       389

    accuracy                         0.9949     27561
   macro avg     0.9891    0.9954    0.9922     27561
weighted avg     0.9950    0.9949    0.9949     27561



---

## Section 9 — Confusion Matrix & Training Curves

In [17]:
# ── Confusion matrix ──
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm,
    annot       = True,
    fmt         = 'd',
    cmap        = 'Blues',
    xticklabels = list(range(10)),
    yticklabels = list(range(10)),
    linewidths  = 0.5,
    ax          = ax
)
ax.set_title(
    'MyanNet — Confusion Matrix (Burmese Digits 0–9)\n'
    f'Test Accuracy: {test_acc*100:.2f}%  |  5,418 trainable params',
    fontsize=13, fontweight='bold', pad=14
)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label',      fontsize=11)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')

Saved: confusion_matrix.png


In [18]:
# ── Training curves ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Accuracy
axes[0].plot(history_final.history['accuracy'],
             color='steelblue',  linewidth=2, label='Train')
axes[0].plot(history_final.history['val_accuracy'],
             color='darkorange', linewidth=2, linestyle='--', label='Validation')
axes[0].set_title('MyanNet — Accuracy', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history_final.history['loss'],
             color='steelblue',  linewidth=2, label='Train')
axes[1].plot(history_final.history['val_loss'],
             color='darkorange', linewidth=2, linestyle='--', label='Validation')
axes[1].set_title('MyanNet — Loss', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('MyanNet Training Curves (with augmentation + LR scheduling)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

Saved: training_curves.png


In [19]:
# ── Misclassified samples ──
wrong_idx = np.where(y_pred != y_test)[0]
n_show    = min(12, len(wrong_idx))

print(f'Misclassified : {len(wrong_idx):,} / {len(y_test):,}')
print(f'Error rate    : {len(wrong_idx)/len(y_test)*100:.3f}%')

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
axes = axes.flatten()

for i, idx in enumerate(wrong_idx[:n_show]):
    axes[i].imshow(X_test[idx].reshape(28, 28), cmap='gray')
    axes[i].set_title(
        f'True: {y_test[idx]}  Pred: {y_pred[idx]}',
        fontsize=9, color='red'
    )
    axes[i].axis('off')

for j in range(n_show, len(axes)):
    axes[j].axis('off')

plt.suptitle('Misclassified Samples — MyanNet', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('misclassified_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: misclassified_samples.png')

Misclassified : 140 / 27,561
Error rate    : 0.508%
Saved: misclassified_samples.png


---

## Section 10 — Model Comparison

In [20]:
# ── Train baseline and GAP-BN for comparison (10 epochs each) ──
def quick_train(model, X_tr, y_tr, X_val, y_val, epochs=10):
    model.fit(
        X_tr, y_tr,
        epochs           = epochs,
        validation_data  = (X_val, y_val),
        batch_size       = BATCH_SIZE,
        verbose          = 0
    )
    _, acc = model.evaluate(X_val, y_val, verbose=0)
    return acc

print('Training Baseline CNN...')
baseline_model = build_baseline()
baseline_acc = quick_train(baseline_model, X_train, y_train,
                            X_test, y_test)
print(f'  Baseline accuracy : {baseline_acc:.4f}')

print('Training GAP-BN CNN...')
gap_model = build_gap_bn()
gap_acc = quick_train(gap_model, X_train, y_train,
                      X_test, y_test)
print(f'  GAP-BN accuracy   : {gap_acc:.4f}')

# MyanNet (already trained above)
myannet_acc = test_acc
print(f'  MyanNet accuracy  : {myannet_acc:.4f}')

Training Baseline CNN...


I0000 00:00:1776071359.151781      67 service.cc:152] XLA service 0x44a97790 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776071359.151841      67 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1776071360.473416      67 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  Baseline accuracy : 0.9958
Training GAP-BN CNN...
  GAP-BN accuracy   : 0.9951
  MyanNet accuracy  : 0.9949


In [21]:
# ── Comparison table ──
def count_inference_params(model):
    t  = sum(int(np.prod(v.shape)) for v in model.trainable_weights)
    nt = sum(int(np.prod(v.shape)) for v in model.non_trainable_weights)
    return t, nt, t + nt

bl_t, bl_nt, bl_inf = count_inference_params(baseline_model)
gp_t, gp_nt, gp_inf = count_inference_params(gap_model)
mn_t, mn_nt, mn_inf = count_inference_params(best_model)

comparison_data = {
    'Model'               : ['Baseline CNN', 'GAP-BN CNN', 'MyanNet (ours)'],
    'Test Accuracy (%)'   : [f'{baseline_acc*100:.2f}', f'{gap_acc*100:.2f}', f'{myannet_acc*100:.2f}'],
    'Trainable Params'    : [f'{bl_t:,}', f'{gp_t:,}', f'{mn_t:,}'],
    'Inference Params'    : [f'{bl_inf:,}', f'{gp_inf:,}', f'{mn_inf:,}'],
    'Key Feature'         : ['Flatten + Dense', 'GAP + BN', 'DS-Conv + GAP + Dropout'],
}

df_compare = pd.DataFrame(comparison_data)
print('═' * 70)
print('  MODEL COMPARISON — Burmese Digit Recognition')
print('═' * 70)
print(df_compare.to_string(index=False))
print('═' * 70)
print(f'\n★  MyanNet achieves highest accuracy with fewest trainable parameters')
print(f'★  Param reduction vs Baseline: {(1 - mn_t/bl_t)*100:.1f}% fewer trainable params')

══════════════════════════════════════════════════════════════════════
  MODEL COMPARISON — Burmese Digit Recognition
══════════════════════════════════════════════════════════════════════
         Model Test Accuracy (%) Trainable Params Inference Params             Key Feature
  Baseline CNN             99.58           34,826           34,826         Flatten + Dense
    GAP-BN CNN             99.51           21,418           21,610                GAP + BN
MyanNet (ours)             99.49           10,634           11,018 DS-Conv + GAP + Dropout
══════════════════════════════════════════════════════════════════════

★  MyanNet achieves highest accuracy with fewest trainable parameters
★  Param reduction vs Baseline: 69.5% fewer trainable params


In [22]:
# ── Side-by-side comparison chart ──
model_names = ['Baseline\nCNN', 'GAP-BN\nCNN', 'MyanNet\n(ours)']
colors      = ['#5b8db8', '#5b8db8', '#e05a2b']
accuracies  = [baseline_acc * 100, gap_acc * 100, myannet_acc * 100]
params      = [bl_inf, gp_inf, mn_inf]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy
bars1 = axes[0].bar(model_names, accuracies,
                    color=colors, edgecolor='white', linewidth=1.2)
axes[0].set_ylim([min(accuracies) - 0.5, 100.3])
axes[0].set_ylabel('Test Accuracy (%)', fontsize=11)
axes[0].set_title('Accuracy Comparison', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars1, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.02,
                 f'{val:.2f}%', ha='center', va='bottom',
                 fontsize=10, fontweight='bold')

# Parameters
bars2 = axes[1].bar(model_names, params,
                    color=colors, edgecolor='white', linewidth=1.2)
axes[1].set_ylabel('Total Inference Parameters', fontsize=11)
axes[1].set_title('Parameter Count Comparison', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(bars2, params):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 100,
                 f'{val:,}', ha='center', va='bottom',
                 fontsize=10, fontweight='bold')

plt.suptitle(
    'Burmese Digit Recognition — Model Comparison\n'
    '(orange = proposed MyanNet)',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_comparison.png')

Saved: model_comparison.png


---

## Section 11 — TFLite Export & Inference Benchmarking

The model is converted to TFLite with **post-training integer quantization** using a representative dataset for calibration, then benchmarked over 1,000 inference runs.

In [23]:
# ── TFLite conversion with post-training quantization ──
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_dataset():
    for i in range(min(500, len(X_train))):
        yield [X_train[i:i+1].astype(np.float32)]

converter.representative_dataset   = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type     = tf.float32
converter.inference_output_type    = tf.float32

try:
    tflite_model = converter.convert()
    print('Full-int8 quantization: OK')
except Exception as e:
    print(f'Full-int8 failed ({e}) — using dynamic-range quantization.')
    converter2 = tf.lite.TFLiteConverter.from_keras_model(best_model)
    converter2.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter2.convert()

TFLITE_PATH = 'myannet_quantized.tflite'
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

model_size_kb = os.path.getsize(TFLITE_PATH) / 1024
print(f'Saved  : {TFLITE_PATH}')
print(f'Size   : {model_size_kb:.2f} KB')

INFO:tensorflow:Assets written to: /tmp/tmp4qm3beab/assets


INFO:tensorflow:Assets written to: /tmp/tmp4qm3beab/assets


Saved artifact at '/tmp/tmp4qm3beab'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='input_layer_31')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  135191992938768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135191961788176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135192157623824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135191961781072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135191961780304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135191961783760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135191961783184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135191989493520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135191961785872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135192157628240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1351921576280

W0000 00:00:1776071445.277213      24 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1776071445.277249      24 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1776071445.290836      24 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled


Full-int8 quantization: OK
Saved  : myannet_quantized.tflite
Size   : 24.18 KB


fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32


In [24]:
# ── Inference benchmarking ──
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

sample = X_test[0:1].astype(np.float32)
BENCHMARK_RUNS = 1000
times = []

for _ in range(BENCHMARK_RUNS):
    interpreter.set_tensor(input_details[0]['index'], sample)
    t0 = time.perf_counter()
    interpreter.invoke()
    times.append(time.perf_counter() - t0)

avg_ms = np.mean(times)  * 1000
p50_ms = np.percentile(times, 50) * 1000
p95_ms = np.percentile(times, 95) * 1000

print('─' * 40)
print(f'Benchmark runs  : {BENCHMARK_RUNS}')
print(f'Mean latency    : {avg_ms:.3f} ms')
print(f'P50 latency     : {p50_ms:.3f} ms')
print(f'P95 latency     : {p95_ms:.3f} ms')
print(f'Model size      : {model_size_kb:.2f} KB')
print('─' * 40)

benchmark_results = {
    'model_size_kb'        : round(model_size_kb, 2),
    'avg_inference_ms'     : round(avg_ms, 4),
    'p50_inference_ms'     : round(p50_ms, 4),
    'p95_inference_ms'     : round(p95_ms, 4),
    'benchmark_runs'       : BENCHMARK_RUNS
}
with open('benchmark_results.json', 'w') as f:
    json.dump(benchmark_results, f, indent=2)
print('Saved: benchmark_results.json')

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


────────────────────────────────────────
Benchmark runs  : 1000
Mean latency    : 0.263 ms
P50 latency     : 0.262 ms
P95 latency     : 0.299 ms
Model size      : 24.18 KB
────────────────────────────────────────
Saved: benchmark_results.json


---

## Section 12 — Results Summary

In [25]:
# ── Print and save full results summary ──
trainable_params = mn_t

summary_lines = [
    '═' * 60,
    '  MyanNet — Final Results Summary',
    f'  Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
    '═' * 60,
    '',
    '── Dataset ──',
    f'  Training samples  : {len(X_train):,}',
    f'  Test samples      : {len(X_test):,}',
    f'  Classes           : {NUM_CLASSES}',
    '',
    '── Best Hyperparameters (Optuna, 30 trials) ──',
    *[f'  {k:<20}: {v}' for k, v in best_params.items()],
    '',
    '── Model ──',
    f'  Architecture      : Depthwise Separable CNN + GAP',
    f'  Trainable params  : {trainable_params:,}',
    f'  Inference params  : {mn_inf:,}',
    '',
    '── Final Test Set Performance ──',
    f'  Test accuracy     : {test_acc:.4f}  ({test_acc*100:.2f}%)',
    f'  Test loss         : {test_loss:.4f}',
    '',
    '── 5-Fold Cross-Validation (test set) ──',
    *[f'  Fold {i+1}             : {acc:.4f}  ({acc*100:.2f}%)'
      for i, acc in enumerate(fold_accuracies)],
    f'  Mean ± Std        : {mean_acc:.4f} ± {std_acc:.4f}',
    '',
    '── TFLite Quantized Model ──',
    f'  Model size        : {model_size_kb:.2f} KB',
    f'  Mean latency      : {avg_ms:.3f} ms / image',
    f'  P95 latency       : {p95_ms:.3f} ms / image',
    '',
    '── Model Comparison ──',
    f'  Baseline CNN      : {baseline_acc*100:.2f}%  ({bl_t:,} trainable params)',
    f'  GAP-BN CNN        : {gap_acc*100:.2f}%  ({gp_t:,} trainable params)',
    f'  MyanNet (ours)    : {myannet_acc*100:.2f}%  ({mn_t:,} trainable params)',
    f'  Param reduction   : {(1 - mn_t/bl_t)*100:.1f}% fewer than Baseline',
    '',
    '── Output Files ──',
    '  myannet_best.keras, myannet_quantized.tflite',
    '  best_params.json, kfold_results.json, benchmark_results.json',
    '  confusion_matrix.png, training_curves.png, model_comparison.png',
    '  kfold_results.png, optuna_results.png, misclassified_samples.png',
    '═' * 60,
]

summary_text = '\n'.join(summary_lines)
print(summary_text)

with open('results_summary.txt', 'w') as f:
    f.write(summary_text + '\n')
print('\nSaved: results_summary.txt')

════════════════════════════════════════════════════════════
  MyanNet — Final Results Summary
  Generated: 2026-04-13 09:10:47
════════════════════════════════════════════════════════════

── Dataset ──
  Training samples  : 60,000
  Test samples      : 27,561
  Classes           : 10

── Best Hyperparameters (Optuna, 50 trials) ──
  filters1            : 64
  filters2            : 64
  dropout             : 0.19369249293034319
  learning_rate       : 0.003004836279033769
  dense_units         : 64

── Model ──
  Architecture      : Depthwise Separable CNN + GAP
  Trainable params  : 10,634
  Inference params  : 11,018

── Final Test Set Performance ──
  Test accuracy     : 0.9949  (99.49%)
  Test loss         : 0.5243

── 5-Fold Cross-Validation (test set) ──
  Fold 1             : 0.9940  (99.40%)
  Fold 2             : 0.9938  (99.38%)
  Fold 3             : 0.9955  (99.55%)
  Fold 4             : 0.9952  (99.52%)
  Fold 5             : 0.9944  (99.44%)
  Mean ± Std        : 0.9946

In [26]:
# ── Verify all output files ──
output_files = [
    'myannet_best.keras',
    'myannet_quantized.tflite',
    'best_params.json',
    'kfold_results.json',
    'benchmark_results.json',
    'results_summary.txt',
    'confusion_matrix.png',
    'training_curves.png',
    'model_comparison.png',
    'kfold_results.png',
    'optuna_results.png',
    'misclassified_samples.png',
    'sample_images.png',
    'class_distribution.png',
    'augmentation_preview.png',
]

print('Output file verification:')
print('─' * 45)
all_ok = True
for fname in output_files:
    if os.path.exists(fname):
        size = os.path.getsize(fname) / 1024
        print(f'  ✓  {fname:<38} ({size:.1f} KB)')
    else:
        print(f'  ✗  {fname}  ← MISSING')
        all_ok = False

print('─' * 45)
print('All files present ✓' if all_ok else 'Some files are missing ✗')

Output file verification:
─────────────────────────────────────────────
  ✓  myannet_best.keras                     (187.5 KB)
  ✓  myannet_quantized.tflite               (24.2 KB)
  ✓  best_params.json                       (0.1 KB)
  ✓  kfold_results.json                     (0.2 KB)
  ✓  benchmark_results.json                 (0.1 KB)
  ✓  results_summary.txt                    (2.1 KB)
  ✓  confusion_matrix.png                   (86.5 KB)
  ✓  training_curves.png                    (90.1 KB)
  ✓  model_comparison.png                   (93.6 KB)
  ✓  kfold_results.png                      (53.9 KB)
  ✓  optuna_results.png                     (109.8 KB)
  ✓  misclassified_samples.png              (51.6 KB)
  ✓  sample_images.png                      (42.1 KB)
  ✓  class_distribution.png                 (62.1 KB)
  ✓  augmentation_preview.png               (31.0 KB)
─────────────────────────────────────────────
All files present ✓
